In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [2]:
connection_url = URL.create(
    "mysql+mysqlconnector",
    username="root",
    password="@sql&form$PHP05",
    host="localhost",
    database="vehicle_sales"
)

In [3]:
engine = create_engine(connection_url)

In [4]:
tables = ['customers', 'inventory_details', 'vehicle_info', 'vehicle_sold']
dfs = {}

for table in tables:
    query = f"SELECT * FROM {table}"
    # Pass the SQLAlchemy 'engine' here instead of the raw connection
    dfs[table] = pd.read_sql(query, con=engine)

In [5]:
# Loop through the dictionary and display the first 5 rows of each table
for table_name, df in dfs.items():
    print(f"--- Table: {table_name} ---")
    display(df.head())  # Using display() makes it look like a clean table in Jupyter
    print("\n") # Adds a bit of space between tables

--- Table: customers ---


,sale_id,customer_name,contact_info
0,S10001,Charles Smith,charles.smith136@gmail.com
1,S10002,Robert Miller,robert.miller944@gmail.com
2,S10003,Anthony Garcia,anthony.garcia502@gmail.com
3,S10004,Matthew Davis,matthew.davis496@gmail.com
4,S10005,Robert Smith,robert.smith942@gmail.com




--- Table: inventory_details ---


,sale_id,region,mileage
0,S1001,North,12000
1,S1002,South,500
2,S1003,West,32000
3,S1004,West,10
4,S1005,West,45000




--- Table: vehicle_info ---


,vehicle_id,make,model,year,body_type,fuel_type,msrp
0,V0001,Unknown,Corolla,2022,Sedan,Gasoline,24000
1,V0002,Ford,F-150,2023,Truck,Diesel,45000
2,V0003,Honda,Civic,2021,Hatchback,Gasoline,22000
3,V0004,Unknown,Model 3,2024,Sedan,Electric,42000
4,V0005,Unknown,X5,2022,SUV,Diesel,62000




--- Table: vehicle_sold ---


,sale_id,vehicle_id,sale_price,days_on_lot
0,S1001,V0001,23200,15
1,S1002,V0002,43500,5
2,S1003,V0003,21000,45
3,S1004,V0004,58000,2
4,S1005,V0005,58000,120


## Remove Duplicates

### 1. Access the customers DataFrame from your dictionary

In [6]:
customer_df = dfs['customers']
customer_df.head()

,sale_id,customer_name,contact_info
0,S10001,Charles Smith,charles.smith136@gmail.com
1,S10002,Robert Miller,robert.miller944@gmail.com
2,S10003,Anthony Garcia,anthony.garcia502@gmail.com
3,S10004,Matthew Davis,matthew.davis496@gmail.com
4,S10005,Robert Smith,robert.smith942@gmail.com


In [7]:
customer_df.shape

(1864, 3)

In [8]:
pd.set_option('display.max.rows', None)
customer_df

,sale_id,customer_name,contact_info
0,S10001,Charles Smith,charles.smith136@gmail.com
1,S10002,Robert Miller,robert.miller944@gmail.com
2,S10003,Anthony Garcia,anthony.garcia502@gmail.com
3,S10004,Matthew Davis,matthew.davis496@gmail.com
4,S10005,Robert Smith,robert.smith942@gmail.com
5,S10006,Mark Gonzalez,mark.gonzalez68@gmail.com
6,S10007,Joseph Garcia,joseph.garcia956@gmail.com
7,S10008,David Gonzalez,david.gonzalez847@gmail.com
8,S10009,Elizabeth Jones,elizabeth.jones706@gmail.com
9,S1001,Thomas Hernandez,thomas.hernandez381@gmail.com


### 2. Get the count before cleaning (to verify it worked)

In [9]:
initial_count = len(customer_df)
initial_count

1864

### 3. Check duplicates

In [10]:
customer_df.duplicated().sum()

np.int64(0)

*Data is already cleaned, we do not need to clean it. You can check
rest of the tables that duplicate values are present there or not.*

### Task 1: Total revenue( in millions)

In [108]:
vehicleSold_df = dfs['vehicle_sold']
total = round(vehicleSold_df['sale_price'].sum()/1000000,2)
total_revenue = f"Total revenue: {total}M"
total_revenue

'Total revenue: 95.53M'

### Task 2: Model with highest sales volume

In [109]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on = 'vehicle_id', how = 'inner')
highest_sales = merge_df.groupby('model')['sale_price'].sum().reset_index()
sales_by_model = highest_sales.rename(columns = {'sale_price': 'highest_sales_revenue'})
top_model = sales_by_model.sort_values(by = 'highest_sales_revenue', ascending = False).head(1)
top_model

,model,highest_sales_revenue
165,Semi,2016000


### Task 3: Average sale price of a vehicle 

In [17]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on = 'vehicle_id', how = 'inner')
average_price = merge_df.groupby('make')['sale_price'].mean().reset_index()
rename_column = average_price.rename(columns = {'sale_price': 'average_sale_price'})
avg_sales = rename_column.sort_values(by = 'average_sale_price', ascending = False)
avg_sales

,make,average_sale_price
3,Tesla,57875.000000
5,Unknown,52650.796569
1,Ford,46218.750000
4,Toyota,39007.692308
2,Honda,29636.000000
0,Chevrolet,22850.000000
6,kia,17600.000000


### Task 4: Vehicle sales count in 'NORTH' region

In [18]:
inventory_df = dfs['inventory_details']
vehicleSold_df = dfs['vehicle_sold']
merge_df = pd.merge(inventory_df, vehicleSold_df, on = 'sale_id', how = 'inner')
north_region = merge_df[merge_df['region'] == 'North']
north_count = north_region.shape[0]
print(f"Count of vehicle sold in North: {north_count}")

Count of vehicle sold in North: 486


### Task 5: What is the most common body_type in the inventory?

In [19]:
vehicleInfo_df = dfs['vehicle_info']
common_type = vehicleInfo_df['body_type'].value_counts().nlargest(1).reset_index()
rename_col = common_type.rename(columns= {'body_type': 'most_common_body'})
rename_col

,most_common_body,count
0,SUV,595


### Task 6: Which region has the highest average days_on_lot?

In [20]:
vehicleSold_df = dfs['vehicle_sold']
inventory_df = dfs['inventory_details']
merge_df = pd.merge(vehicleSold_df, inventory_df, on = 'sale_id', how = 'inner')
merge_df['days_on_lot'] = pd.to_numeric(merge_df['days_on_lot'])
highest_avg = round(merge_df.groupby('region')['days_on_lot'].mean(),2).nlargest(1).reset_index()
rename_col = highest_avg.rename(columns= {'days_on_lot': 'highest_avg_days_on_lot'})
result = rename_col.sort_values(by= 'highest_avg_days_on_lot', ascending = False)
result

,region,highest_avg_days_on_lot
0,East,76.94


### Task 7: What is the average mileage of vehicles sold in the 'WEST' region? 

In [21]:
inventory_df = dfs['inventory_details']
west_region = inventory_df[inventory_df['region'] == 'West']
avg_mileage = round(west_region['mileage'].mean(),2)
print(f"Average mileage of vehicles sold in the 'WEST' region: {avg_mileage}")

Average mileage of vehicles sold in the 'WEST' region: 16034.33


### Task 8: Identify the top 10 customers by total purchase value.

In [22]:
customers_df = dfs['customers']
vehicleSold_df = dfs['vehicle_sold']
merge_df = pd.merge(customers_df, vehicleSold_df, on = 'sale_id', how = 'inner')
total_purchase = merge_df.groupby('customer_name')['sale_price'].sum().nlargest(10).reset_index()
rename_col = total_purchase.rename(columns = {'sale_price': 'total_purchase'})
total_purchase = rename_col.sort_values(by= 'total_purchase', ascending = False)
total_purchase

,customer_name,total_purchase
0,Michael Rodriguez,843000
1,Robert Miller,535000
2,Jessica Hernandez,498000
3,Lisa Garcia,491500
4,Barbara Johnson,454500
5,Betty Garcia,431000
6,Barbara Moore,427000
7,Michael Brown,426000
8,James Wilson,408000
9,Linda Johnson,401500


### Task 9: What is the average discount/premium (Difference between MSRP and Sale Price) for Electric vehicles?

In [23]:
vehicleInfo_df = dfs['vehicle_info']
vehicleSold_df = dfs['vehicle_sold']
merge_df = pd.merge(vehicleInfo_df, vehicleSold_df, on= 'vehicle_id', how = 'inner')
elec_vehicle = merge_df[merge_df['fuel_type'] == 'Electric']
elec_vehicle['difference'] = elec_vehicle['sale_price'] - elec_vehicle['msrp']
elec_vehicle = round(elec_vehicle['difference'].mean(),2)
print(f"Average difference b/w msrp and sale price for electric vehicles is: {elec_vehicle}")


Average difference b/w msrp and sale price for electric vehicles is: 2516.95


### Task 10: Which model year (e.g., 2024 models) is currently selling the fastest (lowest avg days on lot)?

In [24]:
vehicleInfo_df = dfs['vehicle_info']
vehicleSold_df = dfs['vehicle_sold']

# 1. Merge the dataframes
merge_df = pd.merge(vehicleInfo_df, vehicleSold_df, on='vehicle_id', how='inner')

# 2. CONVERT 'days_on_lot' to numeric (This fixes the TypeError)
# errors='coerce' will turn any non-numeric junk into NaN so it doesn't break the math
merge_df['days_on_lot'] = pd.to_numeric(merge_df['days_on_lot'], errors='coerce')

# 3. Calculate the AVERAGE days_on_lot for each year
avg_days_per_year = merge_df.groupby('year')['days_on_lot'].mean().nsmallest(1).reset_index()

rename_col = avg_days_per_year.rename(columns = {'days_on_lot': 'avg_days_on_lot'})

# 4. Sort by days_on_lot (Lowest/fastest at the top)
most_selling_year = rename_col.sort_values(by='avg_days_on_lot', ascending=True)

# 5. Display the result
most_selling_year

,year,avg_days_on_lot
0,2025,1.0


### Task 11: List all vehicle models that sold for more than their MSRP

In [25]:
vehicleInfo_df = dfs['vehicle_info']
vehicleSold_df = dfs['vehicle_sold']
pd.set_option('display.max_rows', None)
# 1. Merge the dataframes
merge_df = pd.merge(vehicleInfo_df, vehicleSold_df, on='vehicle_id', how='inner')
listing_vehicle = merge_df[merge_df['sale_price'] > merge_df['msrp']]
vehicle_sold_more_than_their_msrp = listing_vehicle[['model','msrp', 'sale_price']]
vehicle_sold_more_than_their_msrp.sort_values(by = 'sale_price', ascending = True)

,model,msrp,sale_price
1120,Maverick,25000,29000
494,Maverick,25000,29000
495,Maverick,25000,29000
847,Maverick,26000,30000
1012,Camry,30000,30500
468,BRZ,30000,31000
469,BRZ,30000,31000
844,Maverick,26000,31000
845,Maverick,26000,31000
846,Maverick,26000,31000


### Task 12: Find the total revenue contributed by each fuel_type.

In [26]:
vehicleInfo_df = dfs['vehicle_info']
vehicleSold_df = dfs['vehicle_sold']
pd.set_option('display.max_rows', None)
# 1. Merge the dataframes
merge_df = pd.merge(vehicleInfo_df, vehicleSold_df, on='vehicle_id', how='inner')
total_revenue = merge_df.groupby('fuel_type')['sale_price'].sum().reset_index()
total_revenue_by_each_fuel_type = total_revenue.rename(columns = {'sale_price': 'total_revenue'})
total_revenue_by_each_fuel_type.sort_values(by = 'total_revenue', ascending = False)

,fuel_type,total_revenue
2,Gasoline,54892100
1,Electric,24956000
3,Hybrid,11692100
0,Diesel,3471500
4,Hydrogen,436000
5,PHEV,87000


### Task 13: Calculate the cumulative revenue generated as sale_id increases.

In [27]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on= 'vehicle_id' , how='inner')
merge_df['cumulative_revenue'] = merge_df['sale_price'].cumsum()
cumulative_rev = merge_df[['sale_id','model','sale_price','cumulative_revenue']]
cumulative_rev 

,sale_id,model,sale_price,cumulative_revenue
0,S1001,Corolla,23200,23200
1,S1002,F-150,43500,66700
2,S1003,Civic,21000,87700
3,S1004,Model 3,58000,145700
4,S1005,X5,58000,203700
5,S1006,Accord,27500,231200
6,S1007,Altima,19500,250700
7,S1008,Silverado,58000,308700
8,S1009,A4,36000,344700
9,S1010,Elantra,22500,367200


### Task 14: Which combination of model and body_type yields the highest average profit margin?  

In [28]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on= 'vehicle_id' , how='inner')
merge_df['avg_profit_margin'] = ((merge_df['sale_price'] - merge_df['msrp']) / merge_df['msrp']) * 100
avg_profit = round(merge_df.groupby(['model', 'body_type'])['avg_profit_margin'].mean(),2).reset_index()
avg_profit = avg_profit.sort_values(by = 'avg_profit_margin', ascending = False).head(1)
avg_profit

,model,body_type,avg_profit_margin
24,Blazer,SUV,44.25


### Task 15: Total Profit

In [29]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
total_sale = vehicleSold_df['sale_price'].sum()
total_cost = vehicleInfo_df['msrp'].sum()
total_profit = total_sale - total_cost
print(f"Total profit is: {total_profit}")

Total profit is: 33676200


### Task 16: Gross Profit Margin.

In [30]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
total_sale = vehicleSold_df['sale_price'].sum()
total_cost = vehicleInfo_df['msrp'].sum()
GPM = ((total_sale - total_cost) / total_sale) * 100
print(f"Gross profit margin: {GPM:.2f}%")

Gross profit margin: 35.25%


### Task 17: Compare average days_on_lot for "High Mileage" (>50k) vs "Low Mileage" (<50k) vehicles.

In [31]:
vehicleSold_df = dfs['vehicle_sold']
inventory_df = dfs['inventory_details']
merge_df = pd.merge(vehicleSold_df, inventory_df, on = 'sale_id', how = 'inner')
merge_df['mileage_category'] = merge_df['mileage'].apply(
    lambda x: 'High Mileage' if x > 5000 else 'Low Mileage')

result = merge_df.groupby('mileage_category')['days_on_lot'].mean().reset_index()
# Add this to change the name
result = result.rename(columns={'days_on_lot': 'avg_days_on_lot'})
result

,mileage_category,avg_days_on_lot
0,High Mileage,77.491690
1,Low Mileage,4.503503


### Task 18: Identify "Stale Inventory": Models that average more than 100 days on the lot.

In [32]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on = 'vehicle_id', how = 'inner')
avg_days = round(merge_df.groupby('model')['days_on_lot'].mean(),2).reset_index()
avg_day_on_lot = avg_days[avg_days['days_on_lot'] > 100]
rename_col = avg_day_on_lot.rename(columns = {'days_on_lot': 'avg_days_on_lot'})
rename_col

,model,avg_days_on_lot
2,370Z,150.00
6,6,101.00
8,A-Class,102.00
13,A8,140.00
15,Altima,149.50
17,Armada,210.00
21,Avalon,112.50
36,Camaro,134.00
57,ES 350,109.17
60,EcoSport,195.00


### Task 19: Rank regions by their "Sales Efficiency" (Revenue divided by average Days on Lot).

In [55]:
vehicleSold_df = dfs['vehicle_sold']
inventory_df = dfs['inventory_details']
pd.options.display.float_format ='{:.2f}'.format
merge_df = pd.merge(vehicleSold_df, inventory_df, on= 'sale_id' , how = 'inner')
region_summary = merge_df.groupby('region').agg(
total_sale = ('sale_price' , 'sum'),
avg_days = ('days_on_lot', 'mean')
).reset_index()
region_summary['sale_efficiency'] = region_summary['total_sale'] / region_summary['avg_days']
ranked_region = region_summary.sort_values(by = 'sale_efficiency', ascending = False)
ranked_region = ranked_region[['region', 'sale_efficiency']]
ranked_region 

,region,sale_efficiency
3,West,1416400.41
1,North,832998.70
2,South,723456.00
0,East,26324.19


### Task 20:  What percentage of total sales come from Hybrid/Electric vehicles vs Gasoline?

In [88]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on= 'vehicle_id' , how = 'inner')
filter_fuel_type = merge_df[merge_df['fuel_type'].isin(['Hybrid', 'Electric', 'Gasoline'])]
fuel_sales = filter_fuel_type.groupby('fuel_type')['sale_price'].sum().reset_index()
total_sales = fuel_sales['sale_price'].sum()
fuel_sales['total_sales_percentage'] = (fuel_sales['sale_price'] / total_sales) *100
fuel_sales

,fuel_type,sale_price,total_sales_percentage
0,Electric,24956000,27.26
1,Gasoline,54892100,59.97
2,Hybrid,11692100,12.77


### Task 21: Identify potential data entry errors where the sale_price is less than 50% of the MSRP.

In [107]:
vehicleSold_df = dfs['vehicle_sold']
vehicleInfo_df = dfs['vehicle_info']
merge_df = pd.merge(vehicleSold_df, vehicleInfo_df, on= 'vehicle_id' , how = 'inner')
msrp = (merge_df['msrp'] * 50) / 100 
less_than_50_percent_of_msrp = merge_df[merge_df['sale_price'] < msrp ]
less_than_50_percent_of_msrp

,sale_id,vehicle_id,sale_price,days_on_lot,make,model,year,body_type,fuel_type,msrp
1174,S10261,V0544,58000,1,Unknown,S-Class,2024,Sedan,Gasoline,118000
1186,S10273,V0556,58000,1,Unknown,GT-R,2024,Coupe,Gasoline,121000
1300,S10387,V0667,58000,1,Unknown,S-Class,2024,Sedan,Gasoline,120000
1312,S10399,V0678,58000,1,Unknown,GT-R,2024,Coupe,Gasoline,123000
